# Learning Engagement and Student Performance

**SE 42 · Data Science Basics**  
**Research focus:** How are students’ online learning activities associated with their academic outcomes in Open University courses?

This notebook analyses the Open University Learning Analytics Dataset (OULAD). Its `studentVle.csv` table contains **10,655,280 daily interaction records and six columns**. The related `studentInfo.csv` table contains **32,593 student-course registrations** across seven course modules. A registration is one student taking one course presentation; it is not necessarily a different person.

The original archive is not included in this repository. Download it from the official [UCI dataset page](https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset) and save it as `data/raw/oulad_uci.zip`. The notebook processes the large interaction table in chunks to reduce memory use.

## Five research questions

1. What are the most common final outcomes in the student-course records?
2. How does recorded online engagement (total clicks) differ across final outcomes?
3. How does average weekly online activity develop during a course for each final outcome?
4. Which types of virtual learning materials receive the most clicks?
5. Does the relationship between online engagement and final outcomes look similar across course modules?

The analysis is descriptive. It can identify associations and patterns, but it cannot show that online activity causes a particular academic outcome.

## 1. Imports and project setup

The notebook can be run from the project root or from the `notebooks` folder.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from collections import defaultdict
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")

# Prefer the extracted project in Downloads when it exists. Otherwise,
# use the folder from which this notebook was opened.
CURRENT = Path.cwd()
DOWNLOADS_PROJECT = Path.home() / "Downloads" / "student-performance-learning-engagement"
if DOWNLOADS_PROJECT.exists():
    PROJECT_DIR = DOWNLOADS_PROJECT
elif CURRENT.name == "notebooks":
    PROJECT_DIR = CURRENT.parent
elif (CURRENT / "notebooks").exists():
    PROJECT_DIR = CURRENT
else:
    PROJECT_DIR = CURRENT

DATA_DIR = PROJECT_DIR / "data" / "raw"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_SOURCE_URL = "https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset"
ZIP_PATH = DATA_DIR / "oulad_uci.zip"

print("Project folder:", PROJECT_DIR)
print("Raw data folder:", DATA_DIR)
print("Expected archive:", ZIP_PATH)

## 2. Download and locate the source files

Download the 44.6 MB archive from the official [UCI OULAD page](https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset). Save the downloaded ZIP file as `oulad_uci.zip` inside the `data/raw` folder shown above. The raw dataset is not included in this GitHub repository. This browser-download step also avoids local Python SSL-certificate problems that can sometimes affect automatic downloads.

The next cell extracts the archive once and locates the three tables needed for this analysis.

In [ ]:
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        "OULAD archive not found. Download it from the official UCI page: "
        f"{DATA_SOURCE_URL}\n"
        f"Save the ZIP as: {ZIP_PATH}"
    )

print(f"Using archive: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size / (1024**2):.1f} MB)")
with ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(DATA_DIR)

# Find CSVs even if the archive contains an extra enclosing folder.
def find_csv(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        matches = [p for p in DATA_DIR.rglob("*.csv") if p.name.lower() == filename.lower()]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} after extracting the official archive.")
    return matches[0]

INFO_PATH = find_csv("studentInfo.csv")
STUDENT_VLE_PATH = find_csv("studentVle.csv")
VLE_PATH = find_csv("vle.csv")

print("Student information:", INFO_PATH.name)
print("VLE interactions:", STUDENT_VLE_PATH.name)
print("VLE material types:", VLE_PATH.name)

## 3. Load the smaller tables and check the dataset structure

Each row in `studentInfo.csv` represents a student-course registration. The interaction table is much larger, so it will be read in chunks in the next section.

In [ ]:
info = pd.read_csv(INFO_PATH)
vle = pd.read_csv(VLE_PATH)

KEYS = ["code_module", "code_presentation", "id_student"]
VLE_KEYS = ["code_module", "code_presentation", "id_site"]

required_info = set(KEYS + ["final_result"])
required_vle = set(VLE_KEYS + ["activity_type"])
if not required_info.issubset(info.columns):
    raise ValueError(f"studentInfo.csv is missing columns: {required_info - set(info.columns)}")
if not required_vle.issubset(vle.columns):
    raise ValueError(f"vle.csv is missing columns: {required_vle - set(vle.columns)}")

info["final_result"] = info["final_result"].fillna("Unknown")
info_lookup = info[KEYS + ["final_result"]].drop_duplicates()
vle_lookup = vle[VLE_KEYS + ["activity_type"]].drop_duplicates()

print("Student-course registration rows:", f"{len(info):,}")
print("Student-course key duplicates:", int(info.duplicated(KEYS).sum()))
print("Number of course modules:", info["code_module"].nunique())
print("Final outcome categories:", sorted(info["final_result"].unique()))
display(info.head())

print("\nMissing values in studentInfo.csv (largest counts first):")
display(info.isna().sum().sort_values(ascending=False).head(10).to_frame("missing values"))

## 4. Process all interaction records in chunks

The published dataset description reports 10,655,280 rows in `studentVle.csv`, with six columns. The code below also counts the rows read from the downloaded file, so the notebook verifies the actual local file rather than relying only on a quoted figure. Aggregates are kept at registration, week, material type and course-outcome level; the 10-million-row table is not loaded into memory all at once.

In [ ]:
CHUNK_SIZE = 250_000

# Running totals. The raw data is large, but these summaries are comparatively small.
registration_clicks = defaultdict(int)
weekly_clicks = defaultdict(int)
activity_clicks = defaultdict(int)
module_outcome_clicks = defaultdict(int)
raw_row_count = 0
unmatched_student_rows = 0

reader = pd.read_csv(STUDENT_VLE_PATH, chunksize=CHUNK_SIZE)
for chunk_number, chunk in enumerate(reader, start=1):
    raw_row_count += len(chunk)
    chunk["sum_click"] = pd.to_numeric(chunk["sum_click"], errors="coerce").fillna(0)
    chunk["date"] = pd.to_numeric(chunk["date"], errors="coerce").fillna(0)

    # Add the final outcome to each interaction using the three registration keys.
    enriched = chunk.merge(info_lookup, on=KEYS, how="left", validate="many_to_one")
    unmatched_student_rows += int(enriched["final_result"].isna().sum())
    enriched["final_result"] = enriched["final_result"].fillna("Unknown")

    # A. Total clicks per student-course registration.
    reg_part = enriched.groupby(KEYS, observed=True)["sum_click"].sum()
    for key, value in reg_part.items():
        registration_clicks[key] += int(value)

    # B. Total clicks per course week and final outcome.
    enriched["week"] = (enriched["date"].clip(lower=0) // 7 + 1).astype(int)
    week_part = enriched.groupby(["final_result", "week"], observed=True)["sum_click"].sum()
    for key, value in week_part.items():
        weekly_clicks[key] += int(value)

    # C. Total clicks by type of VLE material.
    with_types = enriched.merge(vle_lookup, on=VLE_KEYS, how="left", validate="many_to_one")
    with_types["activity_type"] = with_types["activity_type"].fillna("Unknown")
    activity_part = with_types.groupby("activity_type", observed=True)["sum_click"].sum()
    for key, value in activity_part.items():
        activity_clicks[str(key)] += int(value)

    # D. Click totals by module and final outcome.
    module_part = enriched.groupby(["code_module", "final_result"], observed=True)["sum_click"].sum()
    for key, value in module_part.items():
        module_outcome_clicks[key] += int(value)

    if chunk_number % 10 == 0:
        print(f"Processed {raw_row_count:,} interaction rows...")

print("\nTotal interaction rows processed:", f"{raw_row_count:,}")
print("Rows without a matching student-course record:", f"{unmatched_student_rows:,}")
print("Meets the 10,000+ row requirement:", raw_row_count >= 10_000)
if raw_row_count < 10_000:
    raise ValueError("The downloaded interaction file has fewer than 10,000 rows. Check the source archive.")

## 5. Create analysis tables

The main analytical table has one row per student-course registration, with total recorded clicks joined to the final outcome and background variables. This makes comparisons easier while preserving the verified raw interaction-row count above.

In [ ]:
reg_rows = [(*key, value) for key, value in registration_clicks.items()]
reg_clicks = pd.DataFrame(reg_rows, columns=KEYS + ["total_clicks"])
student_summary = info.merge(reg_clicks, on=KEYS, how="left", validate="one_to_one")
student_summary["total_clicks"] = student_summary["total_clicks"].fillna(0)

weekly_summary = pd.DataFrame(
    [(result, week, clicks) for (result, week), clicks in weekly_clicks.items()],
    columns=["final_result", "week", "total_clicks"]
)
outcome_sizes = info.groupby("final_result").size().to_dict()
weekly_summary["registrations"] = weekly_summary["final_result"].map(outcome_sizes)
weekly_summary["clicks_per_registration"] = (
    weekly_summary["total_clicks"] / weekly_summary["registrations"]
)

activity_summary = pd.DataFrame(
    [(activity, clicks) for activity, clicks in activity_clicks.items()],
    columns=["activity_type", "total_clicks"]
).sort_values("total_clicks", ascending=False)

module_summary = pd.DataFrame(
    [(module, result, clicks) for (module, result), clicks in module_outcome_clicks.items()],
    columns=["code_module", "final_result", "total_clicks"]
)
module_sizes = info.groupby(["code_module", "final_result"]).size().rename("registrations").reset_index()
module_summary = module_summary.merge(module_sizes, on=["code_module", "final_result"], how="right")
module_summary["total_clicks"] = module_summary["total_clicks"].fillna(0)
module_summary["mean_clicks_per_registration"] = module_summary["total_clicks"] / module_summary["registrations"]

# Save compact derived tables and the figures, not the original 10.6-million-row file.
student_summary.to_csv(PROJECT_DIR / "outputs" / "student_course_summary.csv", index=False)
weekly_summary.to_csv(PROJECT_DIR / "outputs" / "weekly_engagement_summary.csv", index=False)
activity_summary.to_csv(PROJECT_DIR / "outputs" / "activity_type_summary.csv", index=False)
module_summary.to_csv(PROJECT_DIR / "outputs" / "module_outcome_summary.csv", index=False)

print("Student-course analysis rows:", f"{len(student_summary):,}")
print("Analysis columns:", len(student_summary.columns))
display(student_summary.head())

## 6. Research question 1: What are the most common final outcomes?

The bar chart shows the number of student-course registrations in each final-outcome category.

In [ ]:
outcome_counts = info["final_result"].value_counts().reindex(
    ["Distinction", "Pass", "Fail", "Withdrawn", "Unknown"]
).dropna()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=outcome_counts.index, y=outcome_counts.values, color="#28745e", ax=ax)
ax.set_title("Final outcomes across student-course registrations")
ax.set_xlabel("Final outcome")
ax.set_ylabel("Number of registrations")
ax.tick_params(axis="x", rotation=15)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "01_final_outcomes.png", dpi=160, bbox_inches="tight")
plt.show()

most_common = outcome_counts.idxmax()
most_common_share = outcome_counts.max() / outcome_counts.sum() * 100
print(f"Answer to RQ1: {most_common} is the most common recorded outcome, representing {most_common_share:.1f}% of registrations.")
display(outcome_counts.rename("registrations").to_frame())

## 7. Research question 2: How does online engagement differ across final outcomes?

The box plot compares total clicks per student-course registration. A logarithmic scale is used because click counts are strongly skewed and a small number of registrations can have very high activity.

In [ ]:
order = [x for x in ["Distinction", "Pass", "Fail", "Withdrawn", "Unknown"] if x in student_summary["final_result"].unique()]
plot_data = student_summary.copy()
plot_data["log_clicks"] = np.log1p(plot_data["total_clicks"])

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=plot_data, x="final_result", y="log_clicks", order=order, color="#8eb9a5", ax=ax, showfliers=False)
ax.set_title("Online engagement by final outcome")
ax.set_xlabel("Final outcome")
ax.set_ylabel("log(1 + total clicks per registration)")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "02_engagement_by_outcome.png", dpi=160, bbox_inches="tight")
plt.show()

engagement_by_outcome = student_summary.groupby("final_result")["total_clicks"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)
display(engagement_by_outcome.round(1))
highest_engagement_outcome = engagement_by_outcome["mean"].idxmax()
print(f"Answer to RQ2: {highest_engagement_outcome} has the highest mean click total per registration in this dataset. Compare both means and medians because the distribution is skewed.")

## 8. Research question 3: How does weekly activity develop during a course?

Weekly clicks are divided by the total number of registrations in each outcome category. This includes registrations with no recorded clicks in a given week, so the denominator is the full outcome group rather than only active users.

In [ ]:
weekly_plot = weekly_summary.sort_values("week")
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=weekly_plot, x="week", y="clicks_per_registration", hue="final_result", marker="o", ax=ax)
ax.set_title("Average weekly VLE clicks per registration, by final outcome")
ax.set_xlabel("Course week")
ax.set_ylabel("Clicks per registration")
ax.legend(title="Final outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "03_weekly_engagement.png", dpi=160, bbox_inches="tight")
plt.show()

weekly_overall = weekly_summary.groupby("week").agg(total_clicks=("total_clicks", "sum"), registrations=("registrations", "sum"))
weekly_overall["clicks_per_registration"] = weekly_overall["total_clicks"] / weekly_overall["registrations"]
peak_week = int(weekly_overall["clicks_per_registration"].idxmax())
peak_value = weekly_overall.loc[peak_week, "clicks_per_registration"]
print(f"Answer to RQ3: Week {peak_week} has the highest average recorded activity ({peak_value:.1f} clicks per registration). The line chart shows how this pattern differs by final outcome.")

## 9. Research question 4: Which types of learning materials receive the most clicks?

The `studentVle` interaction records are linked to the `vle` table through the material ID and course presentation. This lets us compare the activity types of materials, such as course pages and other online resources.

In [ ]:
top_activities = activity_summary.head(12).sort_values("total_clicks", ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=top_activities, x="total_clicks", y="activity_type", color="#28745e", ax=ax)
ax.set_title("Most-used VLE material types")
ax.set_xlabel("Total recorded clicks")
ax.set_ylabel("VLE activity type")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "04_activity_types.png", dpi=160, bbox_inches="tight")
plt.show()

if not activity_summary.empty:
    top_activity = activity_summary.iloc[0]["activity_type"]
    top_activity_clicks = int(activity_summary.iloc[0]["total_clicks"])
    print(f"Answer to RQ4: '{top_activity}' receives the most recorded clicks ({top_activity_clicks:,}).")
display(activity_summary.head(12))

## 10. Research question 5: Does the engagement pattern vary across course modules?

The heatmap shows mean clicks per student-course registration for each module and final outcome. Empty cells mean that a particular outcome category is not represented in that module.

In [ ]:
heatmap_data = module_summary.pivot(index="code_module", columns="final_result", values="mean_clicks_per_registration")
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=.5, ax=ax)
ax.set_title("Mean clicks per registration by module and final outcome")
ax.set_xlabel("Final outcome")
ax.set_ylabel("Course module")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "05_module_outcome_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

valid_pairs = module_summary.dropna(subset=["mean_clicks_per_registration"])
if not valid_pairs.empty:
    highest_pair = valid_pairs.loc[valid_pairs["mean_clicks_per_registration"].idxmax()]
    lowest_pair = valid_pairs.loc[valid_pairs["mean_clicks_per_registration"].idxmin()]
    print(f"Answer to RQ5: The highest mean engagement is in module {highest_pair['code_module']} for {highest_pair['final_result']} ({highest_pair['mean_clicks_per_registration']:.1f} clicks per registration).")
    print(f"The lowest mean engagement is in module {lowest_pair['code_module']} for {lowest_pair['final_result']} ({lowest_pair['mean_clicks_per_registration']:.1f} clicks per registration).")
    print("The heatmap helps compare patterns across modules; differences may also reflect course design and cohort size, not only student behaviour.")

## 11. Additional diagram: previous attempts and online engagement

This scatter plot is an additional exploratory view. It shows whether click totals vary with the number of previous attempts. It is not a causal test.

In [ ]:
if "num_of_prev_attempts" in student_summary.columns:
    scatter_data = student_summary.sample(n=min(4000, len(student_summary)), random_state=42).copy()
    scatter_data["log_clicks"] = np.log1p(scatter_data["total_clicks"])
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.scatterplot(data=scatter_data, x="num_of_prev_attempts", y="log_clicks", hue="final_result", palette="Set2", alpha=.45, s=28, ax=ax)
    ax.set_title("Previous attempts and online engagement (sample of registrations)")
    ax.set_xlabel("Number of previous attempts")
    ax.set_ylabel("log(1 + total clicks)")
    ax.legend(title="Final outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "06_attempts_vs_engagement.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("The previous-attempts column is not available in this version of the student table.")

## 12. Overall summary and limitations

The following summary is created from the computed tables above, so it reflects the downloaded data rather than pre-written example findings.

In [ ]:
print("PROJECT SUMMARY")
print("=" * 70)
print(f"The analysis processed {raw_row_count:,} daily VLE interaction records with six source columns.")
print(f"The linked student table contains {len(student_summary):,} student-course registrations across {student_summary['code_module'].nunique()} course modules.")
print(f"The most common final outcome is {most_common} ({most_common_share:.1f}% of registrations).")
print(f"The outcome with the highest mean recorded click total is {highest_engagement_outcome}.")
print(f"The peak average activity week is week {peak_week}.")
if not activity_summary.empty:
    print(f"The most-clicked VLE material type is {top_activity}.")
if not valid_pairs.empty:
    print(f"The largest module/outcome mean-click value is for module {highest_pair['code_module']} and outcome {highest_pair['final_result']}.")

print("\nLIMITATIONS")
print("- The data is observational, so associations do not prove causation.")
print("- A student can appear in more than one course registration; registrations are not unique people.")
print("- Clicks measure recorded interactions, not attention, learning quality or understanding.")
print("- Different modules have different lengths, materials and assessment designs.")
print("- The dataset represents selected Open University courses and presentations, not all universities.")
print("- Students with no recorded clicks are included as zero in registration-level summaries.")

print("\nFIGURES SAVED TO:", FIGURE_DIR)
print("Remember to save the notebook after all cells finish running.")